# Jazan NDVI from Sentinel-2 L2A

Uses `wherobots_open_data.imagery.copernicus_sentinel2_l2a_stac`:

1. Find the clearest recent scene that fully covers the Jazan area
2. Load the red (B04) and NIR (B08) bands, clipped to the area
3. Compute NDVI with scale and offset applied (true surface reflectance)
4. Show the NDVI map and statistics
5. Save the NDVI as a GeoTIFF you can download and open in ArcGIS Pro

Run one cell at a time with **Shift + Enter**. A Micro runtime is enough. **Stop the runtime when you finish.**

In [ ]:
from sedona.spark import *
import os, re

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

CATALOG = "wherobots_open_data"
S2_TABLE = f"{CATALOG}.imagery.copernicus_sentinel2_l2a_stac"

# Same Jazan box as Jazan_Test (lon/lat, EPSG:4326)
XMIN, YMIN, XMAX, YMAX = 42.50, 16.85, 42.65, 16.95
AOI_WKT = (f"POLYGON(({XMIN} {YMIN}, {XMAX} {YMIN}, {XMAX} {YMAX}, "
           f"{XMIN} {YMAX}, {XMIN} {YMIN}))")
AOI = f"ST_GeomFromWKT('{AOI_WKT}')"

# Scene search settings: change these to look at another season or year
START, END = "2025-01-01", "2026-12-31"
MAX_CLOUD = 10   # percent

## 1. Find the clearest scene over Jazan

Lists the 10 clearest scenes that fully contain the area, then picks the first one. To use a different scene, change `pick` to another row number.

In [ ]:
from pyspark.sql import functions as F

s2 = sedona.table(S2_TABLE)

scenes = (s2.where(f"year >= {START[:4]} AND year <= {END[:4]}")
            .where(f"datetime >= '{START}' AND datetime <= '{END}'")
            .where(f"`eo:cloud_cover` < {MAX_CLOUD}")
            .where(f"ST_Contains(geometry, {AOI})")
            .selectExpr(
                "datetime",
                "`eo:cloud_cover` AS cloud",
                "mgrs_tile",
                "`proj:epsg` AS epsg",
                "`s2:processing_baseline` AS baseline",
                "assets.red.href AS red_href",
                "assets.nir.href AS nir_href",
                "assets.red.`raster:bands`[0].scale AS scale",
                "assets.red.`raster:bands`[0].offset AS offset",
            )
            .orderBy(F.col("cloud").asc(), F.col("datetime").desc())
            .limit(10)
            .collect())

if not scenes:
    raise ValueError("No scene found. Raise MAX_CLOUD or widen START/END, then rerun.")

for i, s in enumerate(scenes):
    print(f"{i}: {s['datetime']}  cloud={s['cloud']:.2f}%  tile={s['mgrs_tile']}  "
          f"EPSG={s['epsg']}  baseline={s['baseline']}")

pick = 0
scene = scenes[pick]
print("\nUsing scene", pick, "| red:", scene["red_href"])

## 2. Load the red and NIR bands, clipped to Jazan

The bands are Cloud Optimized GeoTIFFs, so only the pixels inside the area are read. The area is reprojected to the scene's UTM zone before clipping.

In [ ]:
EPSG = scene["epsg"]
AOI_UTM = f"ST_SetSRID(ST_Transform({AOI}, 'EPSG:4326', 'EPSG:{EPSG}'), {EPSG})"

def s3_path(href):
    # https://bucket.s3.region.amazonaws.com/key  ->  s3://bucket/key
    m = re.match(r"https://([^./]+)\.s3[.-](?:[a-z0-9-]+\.)?amazonaws\.com/(.+)", href)
    return f"s3://{m.group(1)}/{m.group(2)}" if m else href

def load_clipped(href, name):
    last_err = None
    for p in dict.fromkeys([s3_path(href), href]):
        try:
            df = sedona.sql(f"SELECT RS_Clip(RS_FromPath('{p}'), 1, {AOI_UTM}, true) AS {name}")
            df = df.cache()
            df.first()
            print(f"Loaded {name}:", p)
            return df
        except Exception as e:
            last_err = e
            print(f"Could not load {name} from {p}:", str(e)[:300])
    raise last_err

red = load_clipped(scene["red_href"], "red")
nir = load_clipped(scene["nir_href"], "nir")

red.selectExpr("RS_Width(red) AS width", "RS_Height(red) AS height", "RS_SRID(red) AS srid").show()

## 3. Compute NDVI

Reflectance = DN × scale + offset (offset is −0.1 for processing baseline 04.00 and later). Pixels with DN 0 (no data) or reflectance ≤ 0 after the offset (very dark water or shadow) become −9999, which keeps NDVI between −1 and 1.

In [ ]:
scale = scene["scale"] if scene["scale"] is not None else 0.0001
offset = scene["offset"] if scene["offset"] is not None else 0.0
print("scale:", scale, "| offset:", offset)

script = (f"r = rast0[0] * {scale} + ({offset}); "
          f"n = rast1[0] * {scale} + ({offset}); "
          f"out = con(rast0[0] == 0 || rast1[0] == 0 || r <= 0 || n <= 0, -9999, (n - r) / (n + r));")

ndvi = (red.crossJoin(nir)
           .selectExpr(f"RS_MapAlgebra(red, nir, 'D', '{script}', -9999) AS ndvi")
           .cache())
ndvi.selectExpr("RS_Width(ndvi) AS width", "RS_Height(ndvi) AS height").show()

## 4. NDVI map and statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

row = ndvi.selectExpr("RS_Width(ndvi) AS w", "RS_Height(ndvi) AS h",
                      "RS_BandAsArray(ndvi, 1) AS arr").first()
a = np.array(row["arr"], dtype="float64").reshape(row["h"], row["w"])
a[a == -9999] = np.nan
v = a[~np.isnan(a)]

print(f"Valid pixels: {v.size:,}")
print(f"NDVI min {v.min():.3f} | mean {v.mean():.3f} | max {v.max():.3f}")
print(f"Share of pixels with NDVI > 0.3 (vegetation): {100 * (v > 0.3).mean():.1f}%")
print(f"Share of pixels with NDVI < 0 (mostly water): {100 * (v < 0).mean():.1f}%")

plt.figure(figsize=(10, 7))
plt.imshow(a, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
plt.colorbar(label="NDVI", shrink=0.8)
plt.title(f"Jazan NDVI, Sentinel-2 L2A, {scene['datetime']:%Y-%m-%d} "
          f"(cloud {scene['cloud']:.1f}%)")
plt.axis("off")
plt.show()

## 5. Save the NDVI as a GeoTIFF

Saved to your Wherobots storage. Download it from **Cloud Storage** in the left sidebar of the Wherobots console and open it in ArcGIS Pro or QGIS.

In [ ]:
out_dir = os.getenv("USER_S3_PATH") + "jazan_ndvi"

(ndvi.selectExpr("RS_AsGeoTiff(ndvi) AS tif")
     .write.format("raster")
     .option("rasterField", "tif")
     .option("fileExtension", ".tif")
     .mode("overwrite")
     .save(out_dir))

print("Saved to:", out_dir)

## Done

1. Press **Cmd + S** to save.
2. Go back to the **Code** page and stop the runtime (trash icon under Notebook Runtimes).

Tip: Jazan city is mostly urban, so NDVI will be low. To see farmland, move the box east toward Abu Arish or Sabya by changing `XMIN, YMIN, XMAX, YMAX` in the first cell.